# 12 · Lexicon note review (post-landmark text)

A reading aid, not an analysis. It finds every post-landmark **Clinician, Imaging and Pathology** note of the scored
patients that matches any curated lexicon feature from notebooks 13 / 14 (same patterns, copied verbatim, including the
Gleason-sum bins), so you can compare patients with and without a given term and read the notes behind it.

- **Scores:** the platinum-trained, transferred XGBoost score (`transfer_scores.csv` from notebook 04) at ADT + 180 d.
- **Window:** ADT + landmark < note date ≤ ADT + `WINDOW_MAX_DAYS` (3650 d), as in 13 / 14. Undated notes are dropped.
- **`lexicon_notes`:** one row per matching note, with its matched features, the risk score and the full `RPT_TEXT`.
- **`patients`:** one row per scored patient. It holds the risk score and percentile, the top/bottom-10% group, the note
  counts in the window and one boolean column per lexicon feature (any matching note).

Helpers: `compare(feature)` summarises risk with vs without a feature; `read_notes(feature, ...)` prints matching notes with
the hits highlighted, and `without(feature)` lists patients who have notes but no hit.

**Protected output.** Both tables hold MRNs and raw note text. They are saved as parquet under
`<COMPASS data root>/text_interpretation/post_landmark/lexicon_note_review/`, so keep that folder on the protected data root.


## Configuration


In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import polars as pl

from survival_common import federated_scoring as fs

try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
NOTES_ROOT = Path(os.environ.get("PROFILE_CLINICAL_NOTES", str(Path(cp._PROFILE_DATA_ROOT) / "CLINICAL_NOTES")))
NOTE_PATHS = {
    "Clinician": NOTES_ROOT / "PROGRESS_NOTES.parquet",
    "Imaging": NOTES_ROOT / "IMAGING_NOTES.parquet",
    "Pathology": Path(os.environ.get("PROFILE_PATHOLOGY_NOTES", str(NOTES_ROOT / "PATHOLOGY_NOTES.parquet"))),
}
NOTE_TYPES = ("Clinician", "Imaging", "Pathology")
DATA_PATH = DATA_ROOT / "longitudinal_prediction_data_adt.csv"   # ADT anchor dates (TREATMENT_ANCHOR_DATE)
OUTPUT_DIR = DATA_ROOT / "text_interpretation" / "post_landmark" / "lexicon_note_review"   # protected: MRNs + text

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # risk_score is identical across endpoints
LANDMARK = 180                     # ADT + 180 d, as in 13 / 14
WINDOW_MAX_DAYS = 3650             # post-landmark window cap, as in 13 / 14
RISK_TAIL_FRACTION = 0.10          # top / bottom group labels only
SNIPPET_CHARS = 150                # context either side of a hit in read_notes()
SAVE_PARQUET = True


## Risk scores at the landmark


In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"] == LANDMARK)]
sel = sel[[fs.ID_COL, "risk_score"]].copy()
sel["DFCI_MRN"] = pd.to_numeric(sel.pop(fs.ID_COL), errors="raise").astype("Int64")
assert not sel["DFCI_MRN"].duplicated().any(), "more than one model/arm per patient; narrow the filter above"
sel["risk_score"] = pd.to_numeric(sel["risk_score"], errors="coerce")
sel = sel[np.isfinite(sel["risk_score"])].reset_index(drop=True)
sel["risk_percentile"] = sel["risk_score"].rank(pct=True, method="average")
lo, hi = sel["risk_score"].quantile([RISK_TAIL_FRACTION, 1 - RISK_TAIL_FRACTION])
sel["risk_group"] = np.where(sel["risk_score"] >= hi, "top", np.where(sel["risk_score"] <= lo, "bottom", "middle"))
scored_ids = sorted(int(i) for i in sel["DFCI_MRN"])
print(f"{len(sel):,} scored patients at ADT + {LANDMARK} d; top/bottom cutoffs {lo:.3f} / {hi:.3f}")


## Post-landmark notes

All three note types for the scored patients, kept when ADT + landmark < note date ≤ ADT + `WINDOW_MAX_DAYS`.


In [ ]:
frames = []
for kind in NOTE_TYPES:
    scan = pl.scan_parquet(NOTE_PATHS[kind])
    schema = scan.collect_schema().names()
    cols = [c for c in ("DFCI_MRN", "RPT_ID", "EVENT_DATE", "PROC_DESC", "RPT_TEXT") if c in schema]
    if "EVENT_DATE" not in cols:
        raise ValueError(f"{kind} notes have no EVENT_DATE; the post-landmark window needs dated notes.")
    frame = (scan.select(cols).filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
    frame.insert(1, "NOTE_TYPE", kind)
    frames.append(frame)
all_notes = pd.concat(frames, ignore_index=True)
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes["note_date"] = pd.to_datetime(
    all_notes["EVENT_DATE"].astype("string").str.extract(r"^(\d{4}-\d{2}-\d{2})", expand=False), errors="coerce")
if "RPT_ID" in all_notes:   # one copy per note type x patient x report (as in 14)
    keyed = all_notes["RPT_ID"].notna()
    dup = keyed & all_notes.duplicated(["NOTE_TYPE", "DFCI_MRN", "RPT_ID", "RPT_TEXT", "note_date"])
    all_notes = all_notes[~dup].reset_index(drop=True)

# Post-landmark window, dated from ADT start (TREATMENT_ANCHOR_DATE) as in 13 / 14.
anchor_cols = ["DFCI_MRN", "TREATMENT_ANCHOR_DATE"]
anchor = (pd.read_parquet(DATA_PATH, columns=anchor_cols) if str(DATA_PATH).endswith((".parquet", ".pq"))
          else pd.read_csv(DATA_PATH, usecols=anchor_cols, low_memory=False)).dropna()
anchor["TREATMENT_ANCHOR_DATE"] = pd.to_datetime(anchor["TREATMENT_ANCHOR_DATE"], errors="coerce", format="mixed")
anchor = anchor.dropna().drop_duplicates()
anchor["DFCI_MRN"] = pd.to_numeric(anchor["DFCI_MRN"], errors="raise").astype("int64")
if anchor["DFCI_MRN"].duplicated().any():
    raise ValueError("More than one TREATMENT_ANCHOR_DATE for some patients.")
anchor = anchor.set_index("DFCI_MRN")["TREATMENT_ANCHOR_DATE"]
all_notes["days_from_adt"] = (all_notes["note_date"] - all_notes["DFCI_MRN"].map(anchor)).dt.days.astype(float)
days = all_notes["days_from_adt"]
window_notes = all_notes[(days > LANDMARK) & (days <= WINDOW_MAX_DAYS)].reset_index(drop=True)
print(f"{len(all_notes):,} notes loaded ({int(days.isna().sum()):,} undated); "
      f"{len(window_notes):,} in the post-landmark window from {window_notes['DFCI_MRN'].nunique():,} patients")
display(window_notes.groupby("NOTE_TYPE").agg(notes=("DFCI_MRN", "size"), patients=("DFCI_MRN", "nunique")))


## Curated lexicon

Copied verbatim from notebooks 13 / 14 (themselves copied from `ne_gate.py`). Keep them in sync by hand.


In [ ]:
import re

# Curated lexicon, matched on lowercased report text. The NE patterns are copied verbatim (not imported) from
# LLM_clinical_annotations/tasks/nepc/ne_gate.py (NE_GATE_VERSION "ne-gate-v3", commit
# edb1713ef454f6ece313fa675b5f9438937f831e); update them by hand if that file changes.
NE_TERM = (r"\b(?:neuro[\s-]?endocrine|t-?nepc|nepc|small[\s-]?cell|scpc|scnc|"
           r"oat[\s-]?cell)\b")
SMALL_CELL_TERM = r"\b(?:small[\s-]?cell|scpc|scnc|oat[\s-]?cell)\b"
NE_DIAGNOSIS = (r"\b(?:t-?nepc|nepc|scpc|scnc)\b|"
                r"\b(?:neuro[\s-]?endocrine|small[\s-]?cell|oat[\s-]?cell)"
                r"(?:\s+[\w/()-]+){0,4}?\s+"
                r"(?:carcinoma|cancer|tumou?r|neoplasm|malignancy|component|histology|"
                r"variant|subtype|type|disease|prostate)\b|"
                r"\b(?:carcinoma|cancer)\s+(?:with|of)\s+(?:\w+\s+){0,2}?"
                r"(?:neuro[\s-]?endocrine|small[\s-]?cell)\b")
NE_MARKER = (r"\b(?:synaptophysin|chromogranin(?:\s+a)?|cd\s?56|insm-?1|"
             r"neuron[\s-]+specific\s+enolase|nse)\b")
NON_PROSTATE_NE = (r"\b(?:sclc|small[\s-]?cell\s+lung|merkel|carcinoid|"
                   r"pulmonary\s+(?:origin|primary|small)|lung\s+(?:origin|primary)|"
                   r"urothelial|bladder\s+(?:primary|origin|cancer|carcinoma|tumou?r)|"
                   r"pancreatic\s+(?:net|neuroendocrine)|pnet|gep[\s-]?nets?|"
                   r"well[\s-]differentiated\s+neuroendocrine\s+tumou?r)\b")
NON_SMALL_CELL = r"\bnon[\s-]*$"
# (end of the ne_gate.py copy)

LEXICON = {
    # neuroendocrine (ne_gate.py), then each NE marker from NE_MARKER on its own
    "NE term (any)": NE_TERM,
    "NE diagnosis phrase": NE_DIAGNOSIS,
    "small cell": SMALL_CELL_TERM,
    "NE marker (any)": NE_MARKER,
    "non-prostate NE primary": NON_PROSTATE_NE,
    "synaptophysin": r"\bsynaptophysin\b",
    "chromogranin": r"\bchromogranin\b",
    "CD56": r"\bcd\s?56\b",
    "INSM1": r"\binsm-?1\b",
    "NSE": r"\b(?:neuron[\s-]+specific\s+enolase|nse)\b",
    # other IHC
    "Ki-67 / MIB-1": r"\b(?:ki-?67|mib-?1)\b",
    "AR": r"\b(?:androgen\s+receptor|ar)\b",
    "NKX3.1": r"\bnkx\s?3[.\s-]?1\b",
    "PSMA": r"\b(?:psma|prostate[\s-]specific\s+membrane\s+antigen)\b",
    "ERG": r"\berg\b",
    "PTEN": r"\bpten\b",
    # histology and staging
    "ductal": r"\bductal\b",
    "intraductal": r"\bintraductal\b|\bidc-?p\b",
    "cribriform": r"\bcribriform\b",
    "sarcomatoid": r"\bsarcomatoid\b",
    "treatment effect": r"\b(?:treatment|therapy|hormon\w*|androgen[\s-]deprivation)[\s-]+(?:related\s+)?effects?\b",
    "perineural invasion": r"\bperineural\s+(?:invasion|infiltration)\b|\bpni\b",
    "lymphovascular invasion": (r"\b(?:lympho-?vascular|angiolymphatic|lymphatic|vascular)\s+(?:space\s+)?invasion\b"
                                r"|\blvi\b"),
    "extraprostatic extension": r"\bextra-?prostatic\s+extension\b|\bepe\b",
    "seminal vesicle invasion": r"\bseminal\s+vesicles?\s+(?:invasion|involvement|involved)\b|\bsvi\b",
}
LEXICON_RE = {name: re.compile(rx) for name, rx in LEXICON.items()}
NON_PREFIX_RE = re.compile(NON_SMALL_CELL)
# Gleason: the highest primary + secondary sum in any report (grade group mapped to a sum).
GLEASON_RE = re.compile(r"\bgleason\b[^.;\n]{0,40}?\b([1-5])\s*\+\s*([1-5])\b")
GRADE_GROUP_RE = re.compile(r"\b(?:grade\s+group|isup\s+grade(?:\s+group)?|gg)\s*:?\s*([1-5])\b")
GRADE_GROUP_SUM = {1: 6, 2: 7, 3: 7, 4: 8, 5: 9}
GLEASON_BINS = (("Gleason max 2-6", 2, 6), ("Gleason max 7", 7, 7), ("Gleason max 8-10", 8, 10))

MENTION_FEATURES = list(LEXICON)
LEXICON_FEATURES = MENTION_FEATURES + [label for label, _, _ in GLEASON_BINS]


def lexicon_hits(text):
    """One report's lexicon features: {feature: True} for each mention, plus gleason_max.

    A match right after "non-" ("non-small cell") is not a mention.
    """
    text = text.lower()
    hits = {}
    for name, rx in LEXICON_RE.items():
        if any(not NON_PREFIX_RE.search(text[max(0, m.start() - 8):m.start()]) for m in rx.finditer(text)):
            hits[name] = True
    sums = ([int(a) + int(b) for a, b in GLEASON_RE.findall(text)]
            + [GRADE_GROUP_SUM[int(g)] for g in GRADE_GROUP_RE.findall(text)])
    if sums:
        hits["gleason_max"] = max(sums)
    return hits


## Matching notes and patient table

A note matches when any mention feature hits ("non-" prefixes excluded, as in 13 / 14) or its highest Gleason sum falls in a bin.


In [ ]:
def gleason_bin(g):
    return next((label for label, lo, hi in GLEASON_BINS if lo <= g <= hi), None) if pd.notna(g) else None


note_hits = [lexicon_hits(t) for t in window_notes["RPT_TEXT"].to_numpy()]
flags = pd.DataFrame({f: [bool(h.get(f)) for h in note_hits] for f in MENTION_FEATURES})
gmax = pd.Series([h.get("gleason_max", np.nan) for h in note_hits], dtype=float)
for label, lo_, hi_ in GLEASON_BINS:
    flags[label] = gmax.between(lo_, hi_).to_numpy()
any_hit = flags.any(axis=1).to_numpy()

lexicon_notes = pd.concat([window_notes.drop(columns=["EVENT_DATE"]), flags.astype(bool),
                           gmax.rename("gleason_max")], axis=1)[any_hit]
lexicon_notes.insert(2, "matched_features", flags[any_hit].apply(lambda r: "; ".join(r.index[r]), axis=1))
lexicon_notes = (lexicon_notes.merge(sel, on="DFCI_MRN", how="left", validate="many_to_one")
                 .sort_values(["DFCI_MRN", "note_date"]).reset_index(drop=True))
front = ["DFCI_MRN", "NOTE_TYPE", "note_date", "days_from_adt", "matched_features",
         "risk_score", "risk_percentile", "risk_group"]
lexicon_notes = lexicon_notes[front + [c for c in lexicon_notes if c not in front and c != "RPT_TEXT"] + ["RPT_TEXT"]]

# Patient table: every scored patient, note counts in the window, and any-note presence per feature.
per_patient = pd.concat([window_notes[["DFCI_MRN", "NOTE_TYPE"]], flags], axis=1)
counts = per_patient.groupby("DFCI_MRN").agg(n_notes_in_window=("NOTE_TYPE", "size"))
for kind in NOTE_TYPES:
    counts[f"n_{kind.lower()}_notes"] = per_patient[per_patient.NOTE_TYPE.eq(kind)].groupby("DFCI_MRN").size()
counts["n_lexicon_notes"] = lexicon_notes.groupby("DFCI_MRN").size()
presence = per_patient.groupby("DFCI_MRN")[LEXICON_FEATURES].any()
patients = sel.merge(counts.fillna(0).astype(int).reset_index(), on="DFCI_MRN", how="left")
patients = patients.merge(presence.reset_index(), on="DFCI_MRN", how="left")
num_cols = [c for c in patients if c.startswith("n_")]
patients[num_cols] = patients[num_cols].fillna(0).astype(int)
patients[LEXICON_FEATURES] = patients[LEXICON_FEATURES].fillna(False).astype(bool)
patients["has_notes_in_window"] = patients["n_notes_in_window"] > 0
patients = patients.sort_values("risk_score", ascending=False).reset_index(drop=True)
print(f"lexicon_notes: {len(lexicon_notes):,} matching notes from {lexicon_notes.DFCI_MRN.nunique():,} patients; "
      f"patients: {len(patients):,} scored ({int(patients.has_notes_in_window.sum()):,} with window notes)")


## Helpers and per-feature overview

The overview covers patients with at least one window note. Median percentiles are descriptive only; the tested associations are in 13 / 14.


In [ ]:
def compare(feature, among_with_notes=True):
    """Risk with vs without `feature`; by default only among patients with >= 1 note in the window."""
    p = patients[patients.has_notes_in_window] if among_with_notes else patients
    return (p.groupby(p[feature].map({True: "with", False: "without"}))
             .agg(patients=("DFCI_MRN", "size"), median_score=("risk_score", "median"),
                  median_percentile=("risk_percentile", "median"), mean_percentile=("risk_percentile", "mean"),
                  pct_top=("risk_group", lambda g: g.eq("top").mean()),
                  pct_bottom=("risk_group", lambda g: g.eq("bottom").mean())))


def with_(feature):
    """Patients with >= 1 matching note, highest risk first."""
    return patients[patients[feature]]


def without(feature):
    """Patients with window notes but none matching `feature`, highest risk first."""
    return patients[patients.has_notes_in_window & ~patients[feature]]


def _spans(text, feature):
    low = text.lower()
    if feature in LEXICON_RE:
        return [m.span() for m in LEXICON_RE[feature].finditer(low)
                if not NON_PREFIX_RE.search(low[max(0, m.start() - 8):m.start()])]
    return [m.span() for rx in (GLEASON_RE, GRADE_GROUP_RE) for m in rx.finditer(low)]


def read_notes(feature, mrn=None, note_type=None, n=5, full=False, ascending=False):
    """Print up to `n` notes matching `feature` (highest-risk patients first unless ascending=True).

    Hits are wrapped in >>> <<<. full=True prints the whole note instead of snippets.
    """
    d = lexicon_notes[lexicon_notes[feature]]
    if mrn is not None:
        d = d[d.DFCI_MRN.isin(np.atleast_1d(mrn))]
    if note_type is not None:
        d = d[d.NOTE_TYPE.eq(note_type)]
    d = d.sort_values(["risk_score", "note_date"], ascending=[ascending, True])
    print(f"{feature}: {len(d):,} notes, {d.DFCI_MRN.nunique():,} patients; showing {min(n, len(d))}")
    for _, r in d.head(n).iterrows():
        text, spans = r.RPT_TEXT, _spans(r.RPT_TEXT, feature)
        print("=" * 100)
        print(f"MRN {r.DFCI_MRN} | {r.NOTE_TYPE} | {r.note_date:%Y-%m-%d} (ADT + {r.days_from_adt:.0f} d) | "
              f"score {r.risk_score:.3f} (pct {r.risk_percentile:.2f}, {r.risk_group}) | {r.matched_features}")
        if full:
            for s, e in reversed(spans):
                text = text[:s] + ">>>" + text[s:e] + "<<<" + text[e:]
            print(text)
        else:
            for s, e in spans:
                a, b = max(0, s - SNIPPET_CHARS), min(len(text), e + SNIPPET_CHARS)
                print("  ..." + " ".join((text[a:s] + ">>>" + text[s:e] + "<<<" + text[e:b]).split()) + "...")


p = patients[patients.has_notes_in_window]
overview = pd.DataFrame({f: {"patients_with": int(p[f].sum()), "patients_without": int((~p[f]).sum()),
                             "notes": int(lexicon_notes[f].sum()),
                             "median_pct_with": p.loc[p[f], "risk_percentile"].median(),
                             "median_pct_without": p.loc[~p[f], "risk_percentile"].median()}
                         for f in LEXICON_FEATURES}).T
overview["median_pct_diff"] = overview.median_pct_with - overview.median_pct_without
display(overview.round(3))


## Save (protected)


In [ ]:
if SAVE_PARQUET:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    lexicon_notes.to_parquet(OUTPUT_DIR / "lexicon_notes.parquet", index=False)
    patients.to_parquet(OUTPUT_DIR / "lexicon_patients.parquet", index=False)
    print("wrote (protected: MRNs + note text) ->", OUTPUT_DIR)


## Explore


In [ ]:
# Examples: edit freely.
FEATURE = "NE term (any)"
display(compare(FEATURE))
display(with_(FEATURE).head(10)[["DFCI_MRN", "risk_score", "risk_percentile", "risk_group", "n_notes_in_window", "n_lexicon_notes"]])
display(without(FEATURE).head(10)[["DFCI_MRN", "risk_score", "risk_percentile", "risk_group", "n_notes_in_window"]])
read_notes(FEATURE, n=3)
